<html> <h1 style="font-style:bold; color:blue;"> Neural Computing and Deep Learning </h1> </html>

<html> <h1 style="font-style:italic; color:blue;"> Week-7 </h1> </html>

<html> <h2 style="font-style:italic; color:blue;"> Transfer Learning with Pre-trained CNNs </h2> </html>

Transfer Learning is a machine learning technique where a model trained on one task is re-purposed as the starting point for a model on a different task. Instead of building a CNN from scratch (as in Week 5), we leverage powerful models already trained on millions of images and adapt them to our problem.

### Why Transfer Learning?

- **Saves time**: Pre-trained models already know how to detect edges, textures, shapes and higher-level features.
- **Requires less data**: Very effective even with small labelled datasets.
- **Better performance**: Often outperforms a model trained from scratch on limited data.

**Two common strategies:**
1. **Feature Extraction** – Freeze the pre-trained convolutional base; only train a new classification head.
2. **Fine-Tuning** – Unfreeze some of the top layers of the convolutional base and train them together with the new head at a low learning rate.

<html> <h2 style="font-style:italic; color:blue;"> Task 1 : Feature Extraction with MobileNetV2 </h2> </html>

#### Use MobileNetV2 (pre-trained on ImageNet) as a fixed feature extractor to classify CIFAR-10 images.

### The Data = CIFAR-10

CIFAR-10 is the same dataset used in Week 5.  
It contains **50,000 training** and **10,000 test** colour images (32×32 pixels) across 10 classes:

`airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck`

Because MobileNetV2 expects at least 32×32 input (default 224×224), we will **resize** the images to 96×96 for efficiency.

In [ ]:
# Import Libraries
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.datasets import cifar10

print('TensorFlow version:', tf.__version__)

In [ ]:
# ── 1. Load and preprocess CIFAR-10 ──────────────────────────────────────────
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

# Class names for visualisation
CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

print('Training set shape :', X_train.shape, y_train.shape)
print('Test set shape     :', X_test.shape,  y_test.shape)

In [ ]:
# Visualise a sample of training images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i])
    ax.set_title(CLASS_NAMES[y_train[i][0]])
    ax.axis('off')
plt.suptitle('CIFAR-10 Sample Images', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── 2. Resize images to 96×96 and scale to [0, 1] ────────────────────────────
IMG_SIZE = 96

def preprocess(images):
    """Resize a batch of CIFAR-10 images and scale pixel values to [0, 1]."""
    images = tf.cast(images, tf.float32) / 255.0
    images = tf.image.resize(images, [IMG_SIZE, IMG_SIZE])
    return images

# Work with a subset for faster training in a lab environment
TRAIN_SIZE = 10000
TEST_SIZE  = 2000

X_train_resized = preprocess(X_train[:TRAIN_SIZE]).numpy()
X_test_resized  = preprocess(X_test[:TEST_SIZE]).numpy()

y_train_sub = y_train[:TRAIN_SIZE]
y_test_sub  = y_test[:TEST_SIZE]

print('Resized training shape:', X_train_resized.shape)
print('Resized test shape    :', X_test_resized.shape)

### Step 1 – Load MobileNetV2 (Feature Extraction)

We load MobileNetV2 with `include_top=False` so we discard the original ImageNet classification head.  
Setting `trainable = False` **freezes** all the pre-trained weights.

In [ ]:
# ── 3. Build Feature-Extraction model ────────────────────────────────────────
base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,      # remove the ImageNet classification head
    weights='imagenet'      # use pre-trained ImageNet weights
)

# Freeze the convolutional base – we will NOT update these weights
base_model.trainable = False

print('Base model layers :', len(base_model.layers))
print('Trainable params  :', base_model.count_params())

In [ ]:
# Add a new classification head on top of the frozen base
inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)   # training=False keeps BN layers frozen
x = layers.GlobalAveragePooling2D()(x)   # flatten spatial dimensions
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(10, activation='softmax')(x)  # 10 CIFAR-10 classes

model_fe = keras.Model(inputs, outputs, name='MobileNetV2_FeatureExtraction')
model_fe.summary()

In [ ]:
# Compile the model
model_fe.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train – only the new head is updated
history_fe = model_fe.fit(
    X_train_resized, y_train_sub,
    epochs=10,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

In [ ]:
# Evaluate on the test set
loss_fe, acc_fe = model_fe.evaluate(X_test_resized, y_test_sub, verbose=0)
print(f'Feature Extraction – Test Accuracy: {acc_fe:.4f}  |  Test Loss: {loss_fe:.4f}')

In [ ]:
# Plot training curves
def plot_history(history, title='Training History'):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

    ax1.plot(history.history['accuracy'],     label='Train Accuracy')
    ax1.plot(history.history['val_accuracy'], label='Val Accuracy')
    ax1.set_title('Accuracy')
    ax1.set_xlabel('Epoch')
    ax1.legend()

    ax2.plot(history.history['loss'],     label='Train Loss')
    ax2.plot(history.history['val_loss'], label='Val Loss')
    ax2.set_title('Loss')
    ax2.set_xlabel('Epoch')
    ax2.legend()

    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

plot_history(history_fe, title='Feature Extraction Training')

<html> <h2 style="font-style:italic; color:blue;"> Task 2 : Fine-Tuning </h2> </html>

#### Unfreeze the top layers of MobileNetV2 and fine-tune them with a very low learning rate.

### Step 2 – Fine-Tuning

After training the new head, we can **unfreeze** part of the base model and continue training at a much lower learning rate.  
This allows the pre-trained features to be **gently adjusted** for our specific dataset.

> ⚠️ Always use a **very low learning rate** for fine-tuning to avoid destroying the pre-trained weights.

In [ ]:
# ── 4. Fine-Tuning: unfreeze the top 30 layers of the base model ─────────────
base_model.trainable = True

# How many layers to freeze (keep the early, general-purpose layers frozen)
FINE_TUNE_FROM = len(base_model.layers) - 30

for layer in base_model.layers[:FINE_TUNE_FROM]:
    layer.trainable = False

trainable_count = sum(1 for l in base_model.layers if l.trainable)
print(f'Trainable layers in base model: {trainable_count} / {len(base_model.layers)}')

In [ ]:
# Re-compile with a low learning rate
model_fe.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),  # 100× lower than feature-extraction phase
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Continue training
history_ft = model_fe.fit(
    X_train_resized, y_train_sub,
    epochs=10,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

In [ ]:
# Evaluate after fine-tuning
loss_ft, acc_ft = model_fe.evaluate(X_test_resized, y_test_sub, verbose=0)
print(f'Fine-Tuned Model   – Test Accuracy: {acc_ft:.4f}  |  Test Loss: {loss_ft:.4f}')
print(f'Feature Extraction – Test Accuracy: {acc_fe:.4f}')
print(f'Improvement        : {(acc_ft - acc_fe)*100:+.2f}%')

In [ ]:
plot_history(history_ft, title='Fine-Tuning Training')

<html> <h2 style="font-style:italic; color:blue;"> Task 3 : Predictions and Visualisation </h2> </html>

#### Examine which images the fine-tuned model gets right and wrong.

In [ ]:
# ── 5. Predict and visualise results ─────────────────────────────────────────
y_pred_probs = model_fe.predict(X_test_resized[:20])
y_pred       = np.argmax(y_pred_probs, axis=1)
y_true       = y_test_sub[:20].flatten()

fig, axes = plt.subplots(2, 10, figsize=(18, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_test[i])   # show original (32×32) for clarity
    colour = 'green' if y_pred[i] == y_true[i] else 'red'
    ax.set_title(CLASS_NAMES[y_pred[i]], color=colour, fontsize=8)
    ax.axis('off')
plt.suptitle('Predictions (green = correct, red = wrong)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_pred_all  = np.argmax(model_fe.predict(X_test_resized), axis=1)
y_true_all  = y_test_sub.flatten()

cm = confusion_matrix(y_true_all, y_pred_all)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
fig, ax = plt.subplots(figsize=(10, 8))
disp.plot(ax=ax, xticks_rotation=45, colorbar=False)
ax.set_title('Confusion Matrix – Fine-Tuned MobileNetV2', fontsize=13)
plt.tight_layout()
plt.show()

<html> <h2 style="font-style:italic; color:blue;"> Task 4 : Comparison – Scratch CNN vs Transfer Learning </h2> </html>

#### Build a small CNN from scratch on the same data subset and compare its accuracy with the transfer learning approach.

In [ ]:
# ── 6. Baseline CNN trained from scratch ─────────────────────────────────────
model_scratch = keras.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D(),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D(),
    layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D(),
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(10, activation='softmax')
], name='CNN_from_Scratch')

model_scratch.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_scratch.summary()

In [ ]:
history_scratch = model_scratch.fit(
    X_train_resized, y_train_sub,
    epochs=20,
    batch_size=64,
    validation_split=0.2,
    verbose=1
)

loss_scratch, acc_scratch = model_scratch.evaluate(X_test_resized, y_test_sub, verbose=0)
print(f'Scratch CNN        – Test Accuracy: {acc_scratch:.4f}')
print(f'Feature Extraction – Test Accuracy: {acc_fe:.4f}')
print(f'Fine-Tuned TL      – Test Accuracy: {acc_ft:.4f}')

In [ ]:
# Bar chart comparison
models  = ['Scratch CNN\n(20 epochs)', 'MobileNetV2\nFeature Extraction', 'MobileNetV2\nFine-Tuned']
scores  = [acc_scratch, acc_fe, acc_ft]
colours = ['steelblue', 'darkorange', 'green']

plt.figure(figsize=(8, 5))
bars = plt.bar(models, scores, color=colours, edgecolor='black', width=0.5)
for bar, score in zip(bars, scores):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
             f'{score:.3f}', ha='center', va='bottom', fontweight='bold')
plt.ylim(0, 1.0)
plt.ylabel('Test Accuracy')
plt.title('Model Comparison on CIFAR-10 Subset', fontsize=13)
plt.tight_layout()
plt.show()

### Summary

| Approach | Description | Expected Benefit |
|---|---|---|
| **CNN from Scratch** | All weights randomly initialised | Baseline – needs large dataset & many epochs |
| **Feature Extraction** | Frozen MobileNetV2 + new head | Fast training, good accuracy with little data |
| **Fine-Tuning** | Partially unfrozen MobileNetV2 + new head | Best accuracy; top layers adapt to new domain |

**Key takeaways:**
- Transfer learning with a frozen base trains much faster and achieves competitive accuracy even on small datasets.
- Fine-tuning further improves accuracy by adapting the high-level features of the base model to the target domain.
- Always use a **very low learning rate** during fine-tuning to preserve the useful features learned from ImageNet.
- The choice of *how many layers to unfreeze* is a hyperparameter – experiment to find the best trade-off.

**Further exploration:**
- Try other pre-trained backbones: VGG16, ResNet50, EfficientNetB0.
- Apply data augmentation (random flips, rotations) to improve generalisation.
- Use the full CIFAR-10 training set (50,000 images) for production-quality results.